In [1]:
# Install dependencies (if not already)
# !pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# -----------------------------
# Configuration
# -----------------------------
BASE_URL = "https://www.bbcgoodfood.com"
COLLECTIONS = {
    "dinner": "https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes",
    "lunch": "https://www.bbcgoodfood.com/recipes/collection/easy-lunch-recipes",
    "breakfast": "https://www.bbcgoodfood.com/recipes/collection/breakfast-recipes",
    "batch cooking": "https://www.bbcgoodfood.com/recipes/collection/batch-cooking-recipes",
    "quick and easy": "https://www.bbcgoodfood.com/recipes/collection/quick-and-easy-family-recipes"
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"
}

# -----------------------------
# Functions
# -----------------------------
def get_recipe_links(collection_url):
    """Get all recipe links from a collection page."""
    links = []
    page = 1
    while True:
        url = f"{collection_url}?page={page}"
        print(f"Fetching collection page {page}: {url}")
        res = requests.get(url, headers=headers)
        if res.status_code != 200:
            print("No more pages or failed to fetch collection.")
            break
        soup = BeautifulSoup(res.text, "html.parser")
        recipe_cards = soup.find_all("div", class_="card__section card__content")
        if not recipe_cards:
            break  # No more recipes found
        for card in recipe_cards:
            a_tag = card.find("a", class_="link d-block")
            if a_tag and a_tag.get('href'):
                href = a_tag['href']
                if "/recipes/" in href:
                    if href.startswith("http"):
                        links.append(href)
                    else:
                        links.append(BASE_URL + href)
        page += 1
        time.sleep(1)
    return links

def scrape_recipe(recipe_url):
    """Scrape a single recipe page for title, ingredients, and nutrition info."""
    print(f"Scraping recipe: {recipe_url}")
    res = requests.get(recipe_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    
    # Title
    title_tag = soup.find("h1", class_="post-header__title")
    title = title_tag.text.strip() if title_tag else "No title"
    
    # Ingredients
    ingredients = []
    tabbed_div = soup.find("div", class_="tabbed-list__content")
    if tabbed_div:
        ul_list = tabbed_div.find("ul", class_="ingredients-list")
        if ul_list:
            for li in ul_list.find_all("li", class_="ingredients-list__item"):
                qty_tag = li.find("span", class_="ingredients-list__item-quantity")
                ing_tag = li.find("span", class_="ingredients-list__item-ingredient")
                note_tag = li.find("div", class_="ingredients-list__item-note")
                qty = qty_tag.text.strip() if qty_tag else ""
                ingredient = ing_tag.text.strip() if ing_tag else ""
                note = note_tag.text.strip() if note_tag else ""
                ingredients.append({"quantity": qty, "ingredient": ingredient, "note": note})
    
    # Nutrition
    nutrition = {}
    nutrition_list = soup.find("ul", class_="nutrition-list")
    if nutrition_list:
        for li in nutrition_list.find_all("li", class_="nutrition-list__item"):
            label_tag = li.find("span", class_="nutrition-list__label")
            if label_tag:
                label = label_tag.text.strip()
                value = li.get_text(separator="|").split("|")[1].strip() if "|" in li.get_text(separator="|") else ""
                nutrition[label] = value
    
    return {"title": title, "url": recipe_url, "ingredients": ingredients, "nutrition": nutrition}

def scrape_all_collections(collections):
    """Scrape all recipes and tag them with meal_type."""
    all_recipes = []

    for meal_type, collection_url in collections.items():
        print(f"\n📂 Processing {meal_type}: {collection_url}")
        
        recipe_links = get_recipe_links(collection_url)
        recipe_links = list(set(recipe_links))  # remove duplicates
        print(f"Found {len(recipe_links)} recipes.")

        for link in recipe_links:
            try:
                recipe_data = scrape_recipe(link)
                recipe_data["meal_type"] = meal_type  # ✅ ADD THIS
                all_recipes.append(recipe_data)
                time.sleep(1)
            except Exception as e:
                print(f"❌ Failed: {link} | {e}")

    return all_recipes

# -----------------------------
# Run the scraper
# -----------------------------
recipes = scrape_all_collections(COLLECTIONS)

# -----------------------------
# Save to CSV
# -----------------------------
data_for_csv = []
for r in recipes:
    for ing in r['ingredients']:
        data_for_csv.append({
            "title": r['title'],
            "url": r['url'],
            "meal_type": r["meal_type"],
            "ingredient": ing['ingredient'],
            "quantity": ing['quantity'],
            "note": ing['note'],
            **r['nutrition']
        })

df = pd.DataFrame(data_for_csv)
df.to_csv(r"recipes.csv", index=False)
print("Scraping complete! Data saved to recipes.csv")


📂 Processing dinner: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes
Fetching collection page 1: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=1
Fetching collection page 2: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=2
Fetching collection page 3: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=3
Fetching collection page 4: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=4
Fetching collection page 5: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=5
Fetching collection page 6: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=6
Fetching collection page 7: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=7
Fetching collection page 8: https://www.bbcgoodfood.com/recipes/collection/easy-dinner-recipes?page=8
No more pages or failed to fetch collection.
Found 142 recipes.
Scraping recipe: https://www.bb

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"recipes.csv")

df.head()

df["title"] = (
    df["url"]
    .str.split("/")
    .str[-1]
    .str.replace("-", " ")
    .str.title()
)

df[["title","url"]].head()

df["ingredient_full"] = (
    df["quantity"].fillna("") + " " +
    df["ingredient"].fillna("") + " " +
    df["note"].fillna("")
).str.strip()

ingredients_grouped = (
    df.groupby("url")["ingredient_full"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
)

ingredients_grouped.head()

nutrition_cols = [
    "kcal","fat","saturates","carbs",
    "sugars","fibre","protein","salt"
]

recipes = (
    df.groupby("url")
    .first()
    .reset_index()[["url", "title", "meal_type"] + nutrition_cols]
)

recipes.head()

clean_df = recipes.merge(
    ingredients_grouped,
    on="url",
    how="left"
)

clean_df.head()

clean_df = clean_df.rename(columns={
    "ingredient_full":"ingredients"
})

cols = ["fat","saturates","carbs","sugars","fibre","protein","salt"]

for col in cols:
    clean_df[col] = (
        clean_df[col]
        .astype(str)
        .str.replace(r'(low|medium|high)', '', regex=True)  # remove labels
        .str.extract(r'([\d\.]+)')[0]                       # keep number
        .fillna('')
        + "g"                                               # add grams back
    )
    # Chat wrote this code for cleaning the dataset (title uses the last words of the url, ingredients combines quantity, ingredient and note
    # nutrition removes low/medium/high labels and keeps only numbers with 'g' unit for grams (remove random words like 16glow instead of 16g ))
clean_df.head()

,url,title,meal_type,kcal,fat,saturates,carbs,sugars,fibre,protein,salt,ingredients
0,https://www.bbcgoodfood.com/recipes/15-minute-...,15 Minute Chicken Halloumi Burgers,quick and easy,737.0,42.0g,10.0g,49.0g,6.0g,4.0g,39.0g,4.1g,"2 chicken breast fillets, 1 tbsp oil plus extr..."
1,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Bacon,breakfast,74.0,6.0g,2.0g,0.0g,0.0g,0.0g,6.0g,0.88g,6 rashers streaky bacon or 3 rashers back bacon
2,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Baked Potatoes,quick and easy,206.0,2.0g,0.4g,40.0g,3.0g,5.0g,5.0g,0.01g,"4 baking potatoes (about 250g each), ½ tbsp su..."
3,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Chicken Thighs,quick and easy,111.0,8.0g,2.0g,0.0g,0.0g,0.3g,11.0g,0.7g,"1 tsp paprika, ½ tsp mixed herbs, ½ tsp garlic..."
4,https://www.bbcgoodfood.com/recipes/air-fryer-...,Air Fryer Crispy Chilli Beef,dinner,440.0,19.0g,3.0g,34.0g,20.0g,3.0g,31.0g,3.1g,250g thin-cut minute steak thinly sliced into ...


In [11]:
recipes = []

for url, group in clean_df.groupby("url"):
    recipe = {}

    recipe["title"] = group["title"].iloc[0]
    recipe["url"] = url
    recipe["meal_type"] = group["meal_type"].iloc[0]

    recipe["ingredients"] = [
        {
            "ingredients": row["ingredients"],
        }
        for _, row in group.iterrows()
    ]

    nutrition_cols = ["kcal","fat","saturates","carbs","sugars","fibre","protein","salt"]
    recipe["nutrition"] = group[nutrition_cols].iloc[0].to_dict()

    recipes.append(recipe)

clean_df.to_csv("recipes.csv", index=False)
print("Total recipes:", len(recipes))

Total recipes: 410
